# 01 · Descarga y exploración de datos — SIMEL API

Todos los datos de este proyecto provienen de la API SDMX del SIMEL-INE.
No requiere registro ni credenciales.

**API base:** `https://sdmx.ine.gob.cl/rest/data/CL01,{DATASET},1.0?format=csv`

In [ ]:
import sys
sys.path.insert(0, '../..')  # acceso al cliente compartido

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from simel_client import SIMELClient

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.2f}'.format)

client = SIMELClient()
print('Cliente SIMEL listo.')

## 1. Descarga de todos los datasets del proyecto

In [ ]:
DATASETS = [
    'DF_HRS_TNR_SEXO',          # horas trabajo no remunerado por sexo
    'DF_HRS_TNR_SEXO_QUINTIL',  # por quintil
    'DF_HRS_TNR_SEXO_CLAB',     # por condición laboral
    'DF_HRS_TNR_SEXO_EDAD',     # por tramo etario
    'DF_HRS_TRAB_TOT_SEXO',     # carga global total
    'DF_FDT_SEXO',              # fuerza de trabajo por sexo y región
    'DF_BGYMEDIOOCU',           # brecha salarial por región
]

print('Descargando datos desde SIMEL-INE...')
datos = client.get_multiple(DATASETS)
print(f'\nDatasets descargados: {len(datos)}')

In [ ]:
# Guardar en data/ para no repetir descarga
import os
os.makedirs('../data', exist_ok=True)
for nombre, df in datos.items():
    df.to_csv(f'../data/{nombre}.csv', index=False, encoding='utf-8-sig')
print('Datos guardados en data/')

## 2. El hallazgo central: brecha en horas de cuidado por sexo

In [ ]:
tnr = datos['DF_HRS_TNR_SEXO'].copy()
print('Shape:', tnr.shape)
print('Columnas:', tnr.columns.tolist())
tnr

In [ ]:
# Solo hombres y mujeres (excluir total)
tnr_hm = tnr[tnr['SEXO'].isin(['M', 'F'])].copy()

fig, ax = plt.subplots(figsize=(8, 5))

colores = {'F': '#c0392b', 'M': '#2e86c1'}
anios = sorted(tnr_hm['AÑO'].unique())
x = range(len(anios))
width = 0.35

for i, sexo in enumerate(['F', 'M']):
    vals = tnr_hm[tnr_hm['SEXO'] == sexo].set_index('AÑO')['OBS_VALUE']
    label = 'Mujeres' if sexo == 'F' else 'Hombres'
    ax.bar([xi + i * width for xi in x], [vals.get(a, 0) for a in anios],
           width=width, color=colores[sexo], label=label, edgecolor='white')

ax.set_xticks([xi + width / 2 for xi in x])
ax.set_xticklabels(anios, fontsize=11)
ax.set_ylabel('Horas diarias', fontsize=11)
ax.set_title('Trabajo doméstico y de cuidados no remunerado\nChile 2015 vs 2023', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

# Anotar la brecha
for xi, anio in zip(x, anios):
    f = tnr_hm[(tnr_hm['SEXO'] == 'F') & (tnr_hm['AÑO'] == anio)]['OBS_VALUE'].values
    m = tnr_hm[(tnr_hm['SEXO'] == 'M') & (tnr_hm['AÑO'] == anio)]['OBS_VALUE'].values
    if len(f) and len(m):
        brecha = f[0] / m[0]
        ax.text(xi + width / 2, max(f[0], m[0]) + 0.1,
                f'×{brecha:.1f}', ha='center', fontsize=10, color='#555')

sns.despine()
plt.tight_layout()
plt.savefig('../outputs/figures/tnr_por_sexo.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCambio 2015→2023:')
for sexo in ['F', 'M']:
    vals = tnr_hm[tnr_hm['SEXO'] == sexo].set_index('AÑO')['OBS_VALUE']
    label = 'Mujeres' if sexo == 'F' else 'Hombres'
    if 2015 in vals.index and 2023 in vals.index:
        cambio = vals[2023] - vals[2015]
        print(f'  {label}: {vals[2015]:.2f}h → {vals[2023]:.2f}h ({cambio:+.2f}h)')

## 3. Distribución por quintil: ¿la carga recae más en las más pobres?

In [ ]:
quintil = datos['DF_HRS_TNR_SEXO_QUINTIL'].copy()
print('Columnas:', quintil.columns.tolist())
print('Valores QUINTIL:', quintil['QUINTIL'].unique() if 'QUINTIL' in quintil.columns else 'columna no encontrada')
quintil.head(10)

In [ ]:
# Filtrar mujeres por quintil y comparar con hombres
dim_quintil = 'QUINTIL' if 'QUINTIL' in quintil.columns else quintil.columns[4]  # ajustar según estructura

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, sexo, color, titulo in zip(
    axes,
    ['F', 'M'],
    ['#c0392b', '#2e86c1'],
    ['Mujeres — horas cuidado no remunerado por quintil',
     'Hombres — horas cuidado no remunerado por quintil']
):
    sub = quintil[quintil['SEXO'] == sexo].copy()
    if len(sub) == 0:
        ax.text(0.5, 0.5, 'Sin datos', ha='center', transform=ax.transAxes)
        continue
    pivot = sub.pivot_table(index=dim_quintil, columns='AÑO', values='OBS_VALUE')
    pivot.plot(kind='bar', ax=ax, color=['#aaa', color], edgecolor='white', width=0.7)
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.set_xlabel('Quintil de ingresos')
    ax.set_ylabel('Horas diarias')
    ax.legend(title='Año', fontsize=8)
    ax.tick_params(axis='x', rotation=0)

sns.despine()
plt.tight_layout()
plt.savefig('../outputs/figures/tnr_por_quintil.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Carga global: ¿cuánto trabaja en total cada sexo?

La carga global suma trabajo remunerado + no remunerado. Es la medida más justa de la carga de trabajo total.

In [ ]:
total = datos['DF_HRS_TRAB_TOT_SEXO'].copy()
print('Estructura:')
print(total[total['SEXO'].isin(['M', 'F'])][['SEXO_LABEL', 'AÑO', 'OBS_VALUE', 'UNIDAD']].to_string())

## 5. Relación con participación laboral regional

In [ ]:
fdt = datos['DF_FDT_SEXO'].copy()
print(f'Regiones disponibles: {sorted(fdt["AREA_REF"].unique())}')
print(f'Períodos: {fdt["AÑO"].min()} – {fdt["AÑO"].max()}')
fdt.head()

In [ ]:
# Tasa de participación laboral femenina por región (año más reciente)
anio_max = fdt['AÑO'].max()
fdt_regional = (
    fdt[(fdt['AÑO'] == anio_max) & (fdt['AREA_REF'] != '_T')]
    .pivot_table(index=['REGION', 'AÑO'], columns='SEXO', values='OBS_VALUE')
    .reset_index()
)

if 'F' in fdt_regional.columns and 'M' in fdt_regional.columns:
    fdt_regional['ratio_fm'] = fdt_regional['F'] / fdt_regional['M']

    fig, ax = plt.subplots(figsize=(10, 5))
    fdt_regional_sorted = fdt_regional.sort_values('ratio_fm')
    colores_bar = ['#c0392b' if v < 1 else '#27ae60' for v in fdt_regional_sorted['ratio_fm']]
    ax.barh(fdt_regional_sorted['REGION'], fdt_regional_sorted['ratio_fm'],
            color=colores_bar, edgecolor='white')
    ax.axvline(1, color='gray', linestyle='--', linewidth=1)
    ax.set_xlabel('Ratio fuerza de trabajo Mujeres/Hombres', fontsize=10)
    ax.set_title(f'Participación laboral relativa por región — {anio_max}', 
                 fontsize=12, fontweight='bold')
    sns.despine()
    plt.tight_layout()
    plt.savefig('../outputs/figures/fdt_regional.png', dpi=150, bbox_inches='tight')
    plt.show()